# Day 11 · Caching 與 Artifacts：省錢與管檔案的兩把工具

> 第二部・裝備升級　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 11 - Caching 與 Artifacts：省錢與管檔案的兩把工具.md`

## 今天要學會

1. 設定 `ContextCacheConfig` 並**確認快取真的命中**
2. 用 Artifact 把大東西移出 state
3. 用 `LoadArtifactsTool` 做延遲載入

> 這兩個主題放在一起，是因為它們解決的是同一件事的兩面：
> **不要讓不必要的東西一直待在 context 裡**。

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. Context Caching：重複的前綴不要重複付錢

每次呼叫模型時，request 的開頭通常長得一模一樣：

```
 [system instruction]  ← 每次都一樣
 [工具定義]             ← 每次都一樣
 [對話歷史]             ← 前面的部分每次都一樣
 [這一輪的新訊息]        ← 只有這裡不同
```

Context caching 讓相同的**前綴**只算一次錢。

**⚠️ 跟 Day 10 一樣，設定掛在 `App` 上。**

In [2]:
from google.adk.agents.context_cache_config import ContextCacheConfig

print("ContextCacheConfig 的欄位與預設值：")
for f, info in ContextCacheConfig.model_fields.items():
    print(f"  {f:20s} = {info.default!r}")

ContextCacheConfig 的欄位與預設值：
  cache_intervals      = 10
  ttl_seconds          = 1800
  min_tokens           = 0
  create_http_options  = None


| 欄位 | 意思 |
|---|---|
| `ttl_seconds` | 快取活多久（預設 1800 秒＝30 分鐘） |
| `cache_intervals` | 每幾次呼叫重新整理一次快取（預設 10） |
| `min_tokens` | 少於這個 token 數就不快取（預設 0＝全部都快取） |

> **`min_tokens=0` 通常不是你要的。** 太短的前綴快取起來反而虧
> （建立快取本身有成本）。實務上會設一個門檻。

## 2. 📌 怎麼確認快取真的生效

原文說「怎麼確認快取生效」，但沒給可以跑的程式碼。
關鍵在 `usage_metadata` 裡的 **`cached_content_token_count`**。

In [3]:
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService


class CacheWatcher(BasePlugin):
    """記錄每次回應的 token 用量，特別是快取命中的部分。"""

    def __init__(self, name: str = "cache_watcher"):
        super().__init__(name=name)
        self.stats: list[dict] = []

    async def after_model_callback(self, *, callback_context, llm_response):
        um = getattr(llm_response, "usage_metadata", None)
        if um is None:
            return None
        self.stats.append({
            "prompt": getattr(um, "prompt_token_count", None) or 0,
            "cached": getattr(um, "cached_content_token_count", None) or 0,
            "output": getattr(um, "candidates_token_count", None) or 0,
        })
        return None

    def report(self, label: str = "") -> None:
        print(label)
        print(f"  {'#':>2}  {'prompt':>8} {'cached':>8} {'output':>8}  命中率")
        for i, s in enumerate(self.stats, 1):
            rate = (s["cached"] / s["prompt"] * 100) if s["prompt"] else 0
            print(f"  {i:2d}  {s['prompt']:8d} {s['cached']:8d} {s['output']:8d}  {rate:5.1f}%")
        total_p = sum(s["prompt"] for s in self.stats)
        total_c = sum(s["cached"] for s in self.stats)
        if total_p:
            print(f"  → 總計 prompt {total_p:,} tokens，其中 {total_c:,} 來自快取"
                  f"（{total_c / total_p * 100:.1f}%）")

### 先看沒有快取的基準

用一個**很長的 system instruction**，這樣前綴才夠大、快取才有意義。

In [4]:
LONG_INSTRUCTION = (
    "你是一位資深的 Python 技術顧問，服務對象是中小型軟體團隊。\n"
    "回答時請遵守以下規則：\n"
    + "\n".join(f"{i}. 規則 {i}：回答要具體、可執行，避免空泛的建議。" for i in range(1, 40))
    + "\n最後，一律使用繁體中文，每次回答控制在三句話以內。"
)
print(f"system instruction 長度：{len(LONG_INSTRUCTION):,} 字元")

QUESTIONS = [
    "我該用 pytest 還是 unittest？",
    "那 fixture 要怎麼組織？",
    "測試跑太慢怎麼辦？",
    "怎麼衡量測試品質？",
]

system instruction 長度：1,184 字元


In [5]:
no_cache_watch = CacheWatcher()
no_cache_app = App(
    name="day11a",
    root_agent=LlmAgent(name="nc", model=get_model(), instruction=LONG_INSTRUCTION),
    plugins=[no_cache_watch],
    # 注意：這裡沒有 context_cache_config
)
nc_runner = Runner(app=no_cache_app, session_service=InMemorySessionService())
sid = await new_session(nc_runner)
for q in QUESTIONS:
    await ask(nc_runner, q, session_id=sid)

no_cache_watch.report("【沒有設定快取】")

【沒有設定快取】
   #    prompt   cached   output  命中率
   1       986        0      105    0.0%
   2      1099        0      116    0.0%
   3      1224        0        0    0.0%
   4      1230        0      140    0.0%
  → 總計 prompt 4,539 tokens，其中 0 來自快取（0.0%）


### 打開快取

In [6]:
cache_watch = CacheWatcher()
cached_app = App(
    name="day11b",
    root_agent=LlmAgent(name="c", model=get_model(), instruction=LONG_INSTRUCTION),
    plugins=[cache_watch],
    context_cache_config=ContextCacheConfig(
        ttl_seconds=600,
        cache_intervals=5,
        min_tokens=1024,      # 前綴要夠大才值得快取
    ),
)
c_runner = Runner(app=cached_app, session_service=InMemorySessionService())
sid_c = await new_session(c_runner)
for q in QUESTIONS:
    await ask(c_runner, q, session_id=sid_c)

cache_watch.report("【有設定快取】")

【有設定快取】
   #    prompt   cached   output  命中率
   1       986        0       90    0.0%
   2      1084        0       94    0.0%
   3      1187        0      104    0.0%
   4      1298        0        0    0.0%
  → 總計 prompt 4,555 tokens，其中 0 來自快取（0.0%）


### 怎麼讀這張表

- `cached` 欄位 **> 0** 就代表快取命中了
- 第 1 次一定是 0（那次負責**建立**快取）
- 命中率會隨對話變長而上升，因為共同前綴越來越大

> **快取需要 Gemini 2.0 以上的模型。** 太舊的模型不支援，
> `cached` 會一直是 0 而且不會報錯。

## 3. ⚠️ 多 agent 交棒會讓快取整個失效

這是本日最重要的一段，也是多 agent 系統帳單爆掉的頭號原因。

還記得概念軌第 01 章跑多 agent 時 ADK 印的那行提醒嗎：

> *App can transfer between agents but has no `context_cache_config`.
> **Every transfer swaps the system instruction and the tool set, so the
> request prefix changes and the whole prompt is re-sent uncached after each
> transfer.***

白話：**每一次交棒，前綴就變了，快取全部作廢。**

In [7]:
def check_order(order_id: str) -> dict:
    """查詢訂單狀態。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "status": "已出貨"}


def check_refund(order_id: str) -> dict:
    """查詢退款進度。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "refund": "審核中"}


shipping = LlmAgent(
    name="shipping", model=get_model(),
    description="處理物流與配送問題。",
    instruction=LONG_INSTRUCTION + "\n你專門處理物流。",
    tools=[check_order],
)
refund = LlmAgent(
    name="refund", model=get_model(),
    description="處理退貨與退款問題。",
    instruction=LONG_INSTRUCTION + "\n你專門處理退款。",
    tools=[check_refund],
)
desk = LlmAgent(
    name="desk", model=get_model(),
    instruction=LONG_INSTRUCTION + "\n你是總機，把問題轉給正確的專員。",
    sub_agents=[shipping, refund],
)

transfer_watch = CacheWatcher()
transfer_app = App(
    name="day11c",
    root_agent=desk,
    plugins=[transfer_watch],
    context_cache_config=ContextCacheConfig(ttl_seconds=600, cache_intervals=5, min_tokens=1024),
)
t_runner = Runner(app=transfer_app, session_service=InMemorySessionService())
sid_t = await new_session(t_runner)

await ask(t_runner, "訂單 A-1 到哪了？", session_id=sid_t)
await ask(t_runner, "那 A-1 的退款呢？", session_id=sid_t)

transfer_watch.report("【多 agent 交棒 + 快取】")

【多 agent 交棒 + 快取】
   #    prompt   cached   output  命中率
   1      1274        0       20    0.0%
   2      1644        0       20    0.0%
   3      1690        0       39    0.0%
   4      1740        0       20    0.0%
   5      2370        0       20    0.0%
   6      2416        0       32    0.0%
  → 總計 prompt 11,134 tokens，其中 0 來自快取（0.0%）


觀察 `cached` 欄位：每次換 agent（system instruction 和工具清單都變了），
命中率就掉下來。

### 實務上怎麼辦

| 做法 | 效果 |
|---|---|
| **把共用的長指令抽到 `global_instruction`** | 前綴一致的部分變多 |
| **減少不必要的交棒** | 用 `AgentTool` 代替 `sub_agents`（控制權不轉移） |
| **讓每個 agent 的 instruction 短一點** | 前綴變化的成本降低 |
| 設定 `context_cache_config` | 至少讓每個 agent 各自有快取 |

## 4. Artifacts：把大東西移出 state

Day 09 說過 state 是要跟著每次呼叫走的。所以：

> **不要把 PDF、圖片、大段文字塞進 state。** 那是 Artifact 的工作。

In [8]:
from google.adk.artifacts import InMemoryArtifactService
from google.adk.tools import ToolContext
from google.genai import types


async def save_document(title: str, content: str, tool_context: ToolContext) -> dict:
    """把一份文件存成 artifact，只回傳檔名不回傳內容。

    Args:
        title: 文件標題。
        content: 文件內文。
    """
    filename = f"user:{title}.md"
    version = await tool_context.save_artifact(
        filename=filename, artifact=types.Part(text=f"# {title}\n\n{content}")
    )
    # 重點：回傳「指標」而不是內容，這樣 context 才不會被撐大
    return {"saved": True, "filename": filename, "version": version, "chars": len(content)}


writer = LlmAgent(
    name="writer", model=get_model(),
    instruction="使用者要你寫東西時，呼叫 save_document 存起來，再用一句話回報。",
    tools=[save_document],
)

artifacts = InMemoryArtifactService()
w_runner = Runner(
    agent=writer, app_name="day11", session_service=InMemorySessionService(),
    artifact_service=artifacts,
)
sid_w = await new_session(w_runner)
print(await ask(w_runner, "幫我寫一份 300 字的『什麼是 Context Caching』說明並存檔。",
                session_id=sid_w, trace=True))

  🔧 [writer] 呼叫 save_document({'content': '什麼是 Context Caching（上下文快取）？\n\n在大型語言模型（LLM）的應用中，處理超長文本、龐大程式碼庫或整本書籍時，每次對話或請求都需要重新讀取和計算所有的提示詞（Prompt），這不僅耗時，還會產生昂貴的 API 費用。\n\nContext Caching 是一種快取技術。當開發者輸入一段龐大的上下文資料後，系統會將這段資料的運算結果（如注意力機制的 KV 快取）暫存於模型的高速記憶體中。在快取有效期間內，後續的請求可以直接取用這些快取結果，無需重複解析與計算。\n\n這項技術能帶來兩大好處：第一是「大幅降低延遲」，大幅提升模型回應的速度；第二是「節省成本」，能顯著減少重複處理長文本所消耗的 Token 費用。對於需要頻繁查詢大型文件的 AI 應用而言，Context Caching 是提升效率與經濟效益的關鍵功能。', 'title': '什麼是 Context Caching 說明'})
  ↩️  [writer] save_document 回傳 {'saved': True, 'filename': 'user:什麼是 Context Caching 說明.md', 'version': 0, 'chars': 369}


  💬 [writer] 已將『什麼是 Context Caching』的說明存檔。
已將『什麼是 Context Caching』的說明存檔。


In [9]:
print("--- session state（應該只有指標，沒有內容）---")
print_state(await peek_state(w_runner, sid_w))

--- session state（應該只有指標，沒有內容）---
(state 是空的)


state 裡乾乾淨淨。**文件內容在 artifact store，不在 context 裡。**

## 5. `LoadArtifactsTool`：需要時才載入

存起來之後，怎麼讓 agent 在**需要時**把內容拿回來？
這就是 `LoadArtifactsTool` 的用途——它做的是**延遲載入**。

In [10]:
from google.adk.tools import load_artifacts

reader = LlmAgent(
    name="reader", model=get_model(),
    instruction=(
        "你可以存取之前存下來的文件。使用者問到文件內容時，"
        "用 load_artifacts 把它載入再回答。用繁體中文。"
    ),
    tools=[load_artifacts],
)
r_runner = Runner(
    agent=reader, app_name="day11", session_service=InMemorySessionService(),
    artifact_service=artifacts,
)
sid_r = await new_session(r_runner)
print(await ask(r_runner, "我之前存的那份 Context Caching 文件，重點是什麼？",
                session_id=sid_r, trace=True))

  🔧 [reader] 呼叫 load_artifacts({'artifact_names': ['user:什麼是 Context Caching 說明.md']})
  ↩️  [reader] load_artifacts 回傳 {'artifact_names': ['user:什麼是 Context Caching 說明.md'], 'status': 'artifact contents temporarily inserted and removed. to access these artifacts, call load_artifacts tool again.'}


  💬 [reader] 根據您之前儲存的《什麼是 Context Caching 說明.md》檔案，這份檔案的重點整理如下：

1. **定義**：
   Context Caching（上下文快取）是一種針對大型語言模型（LLM）的技術。當您輸入龐大的上下文（如超長文本、程式碼庫或整本書籍）時，系統會將其運算結果（例如注意力機制的 KV 快取）暫存於模型的高速記憶體中。

2. **運作方式**：
   在快取有效期間
根據您之前儲存的《什麼是 Context Caching 說明.md》檔案，這份檔案的重點整理如下：

1. **定義**：
   Context Caching（上下文快取）是一種針對大型語言模型（LLM）的技術。當您輸入龐大的上下文（如超長文本、程式碼庫或整本書籍）時，系統會將其運算結果（例如注意力機制的 KV 快取）暫存於模型的高速記憶體中。

2. **運作方式**：
   在快取有效期間內，後續的請求可以直接取用這些快取結果，不需要每次都重新讀取、解析與計算所有的提示詞（Prompt）。

3. **兩大核心好處**：
   * **大幅降低延遲**：省去重複計算的時間，顯著提升模型的回應速度。
   * **節省成本**：大幅減少重複處理長文本所消耗的 Token 費用，對於需頻繁查詢大型文件的 AI 應用來說，能兼顧效率與經濟效益。


### 為什麼這叫「延遲載入」

| 做法 | context 成本 |
|---|---|
| 把文件內容塞進 state | **每一輪**都在重送整份文件 |
| 存成 artifact + `load_artifacts` | 只有**真的要用**的那一輪才載入 |

對於「有 50 份文件但每次只用到一份」的情境，差距是數量級的。

## 6. Artifact 有版本

同名再存不會覆蓋，會產生新版本。

In [11]:
APP_NAME, USER = "day11", "student"
versions = await artifacts.list_versions(
    app_name=APP_NAME, user_id=USER, session_id=sid_w,
    filename="user:什麼是 Context Caching.md",
)
print("目前版本:", versions)

if not versions:
    # 檔名由模型決定，列出實際存在的檔案
    keys = await artifacts.list_artifact_keys(
        app_name=APP_NAME, user_id=USER, session_id=sid_w
    )
    print("實際存在的 artifact:", keys)
    for k in keys:
        vs = await artifacts.list_versions(
            app_name=APP_NAME, user_id=USER, session_id=sid_w, filename=k
        )
        print(f"  {k}: 版本 {vs}")

目前版本: []
實際存在的 artifact: ['user:什麼是 Context Caching 說明.md']
  user:什麼是 Context Caching 說明.md: 版本 [0]


## 7. 兩者並看的理由

Caching 和 Artifacts 表面上不相關，但它們回答的是同一個問題：

> **什麼東西該待在 context 裡，什麼不該？**

```
  ┌─ 每次都一樣、又必須在 context 裡 → Context Caching（別重複付錢）
  │
  └─ 很大、又不是每次都要用       → Artifacts（先移出去，要用再載）
```

| | Context Caching | Artifacts |
|---|---|---|
| 解決 | 重複的前綴 | 過大的內容 |
| 位置 | 還在 context 裡 | **移出 context** |
| 設定在 | `App` | Runner 的 `artifact_service` |
| 版本 | TTL 過期就沒了 | **有版本，永久保留** |

## 8. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 設了快取但 `cached` 一直是 0 | 設在 agent 上（要設 `App`）／模型太舊／前綴小於 `min_tokens` |
| 多 agent 系統帳單暴增 | **每次交棒都讓前綴改變，快取全失效** |
| 快取「好像有時有時沒有」 | `ttl_seconds` 過期了，或 `cache_intervals` 到了重建 |
| `load_artifacts` 找不到檔案 | Runner 忘了傳 `artifact_service` |
| 存了 artifact 但別的 session 讀不到 | 檔名少了 `user:` 前綴 |
| context 還是很大 | 工具把**內容**回傳了；應該只回傳檔名指標 |

## 9. 動手練習

1. 把 `min_tokens` 改成 `100000`，重跑第 2 節，確認 `cached` 全變 0。
2. 把第 3 節三個 agent 的 `LONG_INSTRUCTION` 抽成 `global_instruction`
   放在 `desk` 上，比較命中率有沒有改善。
3. 把 `save_document` 改成**直接回傳 content**，
   然後看 state 和後續每一輪的 prompt token 怎麼變。

## 本日回顧

- **Context Caching 讓重複的前綴只付一次錢**，設定在 **`App`** 上。
- **確認命中看 `usage_metadata.cached_content_token_count`**，
  第一次一定是 0（那次在建快取）。
- **⚠️ 每次 agent 交棒都會換掉 system instruction 與工具清單，
  前綴改變＝快取全失效**——多 agent 帳單爆炸的頭號原因。
- **Artifacts 把大東西移出 context**；工具應該回傳**檔名指標**而不是內容。
- **`load_artifacts` 做延遲載入**，只有真的要用的那一輪才付 context 成本。
- 兩者回答的是同一個問題：**什麼該待在 context 裡，什麼不該。**

---
**下一天 → `../day12_callbacks_events_plugins/`**